## Statistical tests

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    ttest_rel,
    wilcoxon,
    shapiro,
    pearsonr,
    spearmanr,
    friedmanchisquare
)

import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.anova import AnovaRM
import scipy.stats as stats
from scipy.stats import shapiro

import pingouin as pg
from statsmodels.stats.multitest import multipletests

In [ ]:
# Project directory
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
     if (p / "notebooks").is_dir()),
    CURRENT_DIR
)

# Project directories
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
RESULTS_DIR = PROJECT_ROOT / "results"
CONFIG_DIR = PROJECT_ROOT / "config"

In [ ]:
outputs_csv_path = PROJECT_ROOT / "outputs" / "pvs_metrics_perc90.csv"

if not outputs_csv_path.is_file():
    raise FileNotFoundError(
        f"Metrics file not found: {outputs_csv_path}"
    )

df_90 = pd.read_csv(outputs_csv_path)

## 1) Pairwise Comparisons

In [ ]:
def run_pairwise_comparisons(
    df,
    features,
    modality_pairs=(("EPC", "T2"), ("EPC", "T1")),
    alpha=0.05,
    subject_col="subject_id"
):
    results = []

    for feature in features:
        for mod1, mod2 in modality_pairs:

            col1 = f"{feature}_{mod1}"
            col2 = f"{feature}_{mod2}"

            # Check required columns
            if col1 not in df.columns or col2 not in df.columns:
                print(f"Skipping {col1} vs {col2}: column not found.")
                continue

            # Keep paired observations
            paired_df = df[[col1, col2]].copy()

            paired_df[col1] = pd.to_numeric(
                paired_df[col1], errors="coerce"
            )
            paired_df[col2] = pd.to_numeric(
                paired_df[col2], errors="coerce"
            )

            paired_df = paired_df.dropna()

            x = paired_df[col1].to_numpy()
            y = paired_df[col2].to_numpy()
            differences = x - y

            n = len(paired_df)

            if n < 3:
                print(f"Skipping {feature}, {mod1} vs {mod2}: insufficient data.")
                continue

            # Check normality of paired differences
            if np.allclose(differences, differences[0]):
                shapiro_stat = np.nan
                shapiro_p = np.nan
                normality = "Constant differences"
                selected_test = "Wilcoxon signed-rank test"

            else:
                shapiro_sample = differences

                if n > 5000:
                    rng = np.random.default_rng(42)
                    shapiro_sample = rng.choice(
                        differences, size=5000, replace=False
                    )

                shapiro_stat, shapiro_p = shapiro(shapiro_sample)

                if shapiro_p >= alpha:
                    normality = "Normal"
                    selected_test = "Paired t-test"
                else:
                    normality = "Non-normal"
                    selected_test = "Wilcoxon signed-rank test"

            # Run selected test
            if selected_test == "Paired t-test":
                test_result = ttest_rel(x, y)
                test_stat = test_result.statistic
                test_p = test_result.pvalue

            else:
                if np.allclose(differences, 0):
                    test_stat = 0
                    test_p = 1.0
                else:
                    test_result = wilcoxon(x, y)
                    test_stat = test_result.statistic
                    test_p = test_result.pvalue

            results.append({
                "Feature": feature,
                "Comparison": f"{mod1} vs {mod2}",
                "N": n,
                "Shapiro_statistic": shapiro_stat,
                "Shapiro_p_value": shapiro_p,
                "Normality": normality,
                "Selected_test": selected_test,
                "Test_statistic": test_stat,
                "p_value": test_p
            })

    return pd.DataFrame(results)

In [ ]:
pairwise_features = [
    "pvs_to_wm_ratio",
    "pvs_wm_contrast_z",
    "pvs_count",
    "mean_diameter_mm",
    "pvsvf_wm",
    "pvs_volume_wm_mm"
]

pairwise_results = run_pairwise_comparisons(
    df=df_90,
    features=pairwise_features
)

display(pairwise_results)

########################################################################


## 2) Calculate Correlation

In [ ]:
def run_correlations(
    df,
    features,
    modality_pairs=(("EPC", "T2"), ("EPC", "T1")),
    alpha=0.05
):
    results = []

    for feature in features:
        for mod1, mod2 in modality_pairs:

            col1 = f"{feature}_{mod1}"
            col2 = f"{feature}_{mod2}"

            if col1 not in df.columns or col2 not in df.columns:
                print(f"Skipping {col1} vs {col2}: column not found.")
                continue

            # Keep paired observations
            paired_df = df[[col1, col2]].copy()

            paired_df[col1] = pd.to_numeric(
                paired_df[col1], errors="coerce"
            )
            paired_df[col2] = pd.to_numeric(
                paired_df[col2], errors="coerce"
            )

            paired_df = paired_df.dropna()

            x = paired_df[col1].to_numpy()
            y = paired_df[col2].to_numpy()

            n = len(paired_df)

            if n < 3:
                print(f"Skipping {feature}, {mod1} vs {mod2}: insufficient data.")
                continue

            # Check for constant variables
            if np.ptp(x) == 0 or np.ptp(y) == 0:
                results.append({
                    "Feature": feature,
                    "Comparison": f"{mod1} vs {mod2}",
                    "N": n,
                    "Shapiro_p_EPC": np.nan,
                    f"Shapiro_p_{mod2}": np.nan,
                    "Selected_test": "Undefined (constant variable)",
                    "Correlation": np.nan,
                    "p_value": np.nan
                })
                continue

            # Shapiro-Wilk tests
            x_sample = x
            y_sample = y

            if n > 5000:
                rng = np.random.default_rng(42)
                indices = rng.choice(n, size=5000, replace=False)
                x_sample = x[indices]
                y_sample = y[indices]

            _, p_x = shapiro(x_sample)
            _, p_y = shapiro(y_sample)

            # Select correlation method
            if p_x >= alpha and p_y >= alpha:
                correlation, p_value = pearsonr(x, y)
                selected_test = "Pearson"

            else:
                correlation, p_value = spearmanr(x, y)
                selected_test = "Spearman"

            results.append({
                "Feature": feature,
                "Comparison": f"{mod1} vs {mod2}",
                "N": n,
                "Shapiro_p_EPC": p_x,
                f"Shapiro_p_{mod2}": p_y,
                "Selected_test": selected_test,
                "Correlation": correlation,
                "p_value": p_value
            })

    return pd.DataFrame(results)

In [ ]:
correlation_features = [
    "pvs_count",
    "mean_diameter_mm",
    "pvsvf_wm",
    "pvs_wm_contrast_z",
    "pvs_volume_wm_mm",
    "pvs_to_wm_ratio"
]

correlation_results = run_correlations(
    df=df_90,
    features=correlation_features
)

display(correlation_results)

In [ ]:
# Save the results of pairwise comparison and correlation
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

pairwise_results.to_csv(results_dir / "pairwise_comparisons.csv", index=False)

correlation_results.to_csv(results_dir / "correlation_results.csv", index=False)

## 3) Running Friedman or ANovaRM across modalities

In [ ]:
def analyze_repeated_measures(
    data,
    metric,
    subject_col="subject_id",
    modalities=("EPC", "T1", "T2"),
    alpha=0.05,
    method="auto",
    make_plots=True
):
    """
    Compare a metric across EPC, T1, and T2.

    Steps:
    1. Complete-case selection
    2. Shapiro-Wilk tests on pairwise differences
    3. Q-Q plots and boxplot
    4. ANOVA or Friedman omnibus test
    5. Sphericity assessment and GG correction for ANOVA
    6. Appropriate pairwise post-hoc tests
    7. Bonferroni correction

    method:
        "auto"      : Select based on pairwise-difference normality.
        "anova"     : Force repeated-measures ANOVA.
        "friedman"  : Force Friedman test.

    Returns:
        summary, normality_results, posthoc_results
    """

    # --------------------------------------------------
    # 1. Check columns and prepare complete-case data
    # --------------------------------------------------

    columns = [f"{metric}_{mod}" for mod in modalities]

    required_columns = [subject_col] + columns

    missing_columns = [
        col for col in required_columns
        if col not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns for {metric}: {missing_columns}"
        )

    if data[subject_col].duplicated().any():
        raise ValueError(
            f"Duplicate subject IDs found for {metric}. "
            "Expected one row per participant."
        )

    subset = data[required_columns].dropna().copy()

    n_subjects = len(subset)

    if n_subjects < 3:
        raise ValueError(
            f"Not enough complete subjects for {metric}: "
            f"n={n_subjects}"
        )

    print("\n" + "=" * 65)
    print(f"METRIC: {metric}")
    print("=" * 65)

    print(f"Complete subjects: {n_subjects}")

    # --------------------------------------------------
    # 2. Examine normality of pairwise differences
    # --------------------------------------------------

    pairs = [(modalities[0], modalities[1]), (modalities[0], modalities[2])]

    normality_records = []

    for mod1, mod2 in pairs:

        col1 = f"{metric}_{mod1}"
        col2 = f"{metric}_{mod2}"

        differences = subset[col1] - subset[col2]

        stat, pval = stats.shapiro(differences)

        normality_records.append({
            "Metric": metric,
            "Comparison": f"{mod1} vs {mod2}",
            "Shapiro_W": stat,
            "Shapiro_p": pval,
            "Normality_supported": pval >= alpha
        })

        print(
            f"{mod1} vs {mod2}: "
            f"Shapiro-Wilk p = {pval:.4f}"
        )

        if make_plots:

            plt.figure(figsize=(5, 5))

            stats.probplot(
                differences,
                dist="norm",
                plot=plt
            )

            plt.title(
                f"{metric}: {mod1} - {mod2}"
            )

            plt.tight_layout()
            plt.show()

    normality_df = pd.DataFrame(normality_records)

    normality_ok = normality_df[
        "Normality_supported"
    ].all()

    # --------------------------------------------------
    # 3. Visual assessment of distributions
    # --------------------------------------------------

    if make_plots:

        plt.figure(figsize=(7, 5))

        plot_data = subset[columns].copy()
        plot_data.columns = list(modalities)

        sns.boxplot(data=plot_data)

        plt.title(f"{metric}: Distribution Across Modalities")
        plt.ylabel(metric)
        plt.tight_layout()
        plt.show()

    # --------------------------------------------------
    # 4. Select statistical method
    # --------------------------------------------------

    if method not in ["auto", "anova", "friedman"]:
        raise ValueError(
            "method must be 'auto', 'anova', or 'friedman'"
        )

    if method == "auto":

        if normality_ok:
            selected_method = "anova"
        else:
            selected_method = "friedman"

    else:
        selected_method = method

    print(f"\nSelected method: {selected_method.upper()}")

    # --------------------------------------------------
    # 5. Run omnibus test
    # --------------------------------------------------

    sphericity_p = np.nan
    correction = "Not applicable"
    effect_size = np.nan

    if selected_method == "anova":

        # Mauchly's test of sphericity
        try:

            spher, W, chi2_spher, dof_spher, sphericity_p = (
                pg.sphericity(subset[columns])
            )

            print(
                f"Sphericity test (Mauchly): "
                f"p = {sphericity_p:.4f}"
            )

        except Exception as error:

            print("Sphericity test could not be completed:", error)

        # Convert to long format
        long_data = subset.melt(
            id_vars=subject_col,
            value_vars=columns,
            var_name="modality",
            value_name="value"
        )

        long_data["modality"] = (
            long_data["modality"]
            .str.replace(f"{metric}_", "", regex=False)
        )

        # Repeated-measures ANOVA
        anova_result = pg.rm_anova(
            data=long_data,
            dv="value",
            within="modality",
            subject=subject_col,
            detailed=True,
            correction=True,
            effsize="np2"
        )

        result_row = anova_result.iloc[0]

        statistic = result_row["F"]

        # Use Greenhouse-Geisser corrected p-value
        # if sphericity is violated.
        if (
            not np.isnan(sphericity_p)
            and sphericity_p < alpha
            and "p-GG-corr" in anova_result.columns
            and pd.notna(result_row["p-GG-corr"])
        ):

            omnibus_p = result_row["p-GG-corr"]
            correction = "Greenhouse-Geisser"

        else:

            omnibus_p = result_row["p-unc"]
            correction = "Uncorrected"

        effect_size = result_row["np2"]

        print("\nRepeated-measures ANOVA:")
        print(anova_result)

        print(f"\nF statistic = {statistic:.4f}")
        print(f"Omnibus p-value = {omnibus_p:.4e}")
        print(f"P-value correction = {correction}")

    else:

        # Friedman test
        values = [
            subset[f"{metric}_{mod}"].values
            for mod in modalities
        ]

        if np.allclose(
            subset[columns].to_numpy(),
            subset[columns[0]].to_numpy()[:, None]
        ):

            statistic = 0.0
            omnibus_p = 1.0

        else:

            statistic, omnibus_p = stats.friedmanchisquare(
                *values
            )

        print("\nFriedman test:")
        print(f"Chi-square = {statistic:.4f}")
        print(f"Omnibus p-value = {omnibus_p:.4e}")

    # --------------------------------------------------
    # 6. Pairwise post-hoc comparisons
    # --------------------------------------------------

    posthoc_records = []

    for mod1, mod2 in pairs:

        col1 = f"{metric}_{mod1}"
        col2 = f"{metric}_{mod2}"

        x = subset[col1]
        y = subset[col2]

        if selected_method == "anova":

            differences = x - y

            if np.allclose(differences, 0):

                test_stat = 0.0
                raw_p = 1.0

            else:

                test_stat, raw_p = stats.ttest_rel(x, y)

            test_name = "Paired t-test"

        else:

            differences = x - y

            if np.allclose(differences, 0):

                test_stat = 0.0
                raw_p = 1.0

            else:

                test_stat, raw_p = stats.wilcoxon(
                    x,
                    y,
                    zero_method="wilcox"
                )

            test_name = "Wilcoxon signed-rank"

        posthoc_records.append({
            "Metric": metric,
            "Comparison": f"{mod1} vs {mod2}",
            "Test": test_name,
            "Statistic": test_stat,
            "Raw_p": raw_p
        })

    posthoc_df = pd.DataFrame(posthoc_records)

    # Bonferroni correction within each metric
    posthoc_df["Adjusted_p"] = multipletests(
        posthoc_df["Raw_p"],
        method="bonferroni"
    )[1]

    posthoc_df["Significant"] = (
        posthoc_df["Adjusted_p"] < alpha
    )

    print("\nPost-hoc comparisons:")
    print(posthoc_df.to_string(index=False))

    # --------------------------------------------------
    # 7. Create summary
    # --------------------------------------------------

    summary = {
        "Metric": metric,
        "N": n_subjects,
        "Selected_test": selected_method,
        "Omnibus_statistic": statistic,
        "Omnibus_p": omnibus_p,
        "Sphericity_p": sphericity_p,
        "ANOVA_correction": correction,
        "Effect_size_np2": effect_size,
        "Significant": omnibus_p < alpha
    }

    summary_df = pd.DataFrame([summary])

    return summary_df, normality_df, posthoc_df


In [ ]:
# Give the metrics to the function
metrics = [
    "pvs_count",
    "mean_diameter_mm",
    "pvs_volume_wm_mm",
    "pvsvf_wm",
    "pvs_to_wm_ratio",
    "pvs_wm_contrast_z"
]

# Store results from all metrics
all_summaries = []
all_normality_results = []
all_posthoc_results = []


# Run the analysis for every metric
for metric in metrics:

    summary, normality, posthoc = analyze_repeated_measures(
        data=df_90,
        metric=metric,
        subject_col="subject_id",
        modalities=("EPC", "T1", "T2"),
        alpha=0.05,
        method="auto",
        make_plots=True
    )

    all_summaries.append(summary)
    all_normality_results.append(normality)
    all_posthoc_results.append(posthoc)


# Combine results into three DataFrames
summary_results = pd.concat(
    all_summaries,
    ignore_index=True
)

normality_results = pd.concat(
    all_normality_results,
    ignore_index=True
)

posthoc_results = pd.concat(
    all_posthoc_results,
    ignore_index=True
)


# Display the final results
print("\n\nFINAL OMNIBUS RESULTS")
display(summary_results)

print("\n\nNORMALITY RESULTS")
display(normality_results)

print("\n\nPOST-HOC RESULTS")
display(posthoc_results)


In [ ]:
# Create output directory
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Save statistical results
summary_results.to_csv(
    results_dir / "repeated_measures_summary.csv",
    index=False
)

normality_results.to_csv(
    results_dir / "normality_results.csv",
    index=False
)

posthoc_results.to_csv(
    results_dir / "posthoc_results.csv",
    index=False
)

print("Statistical results saved successfully.")

## 4) LME models

In [ ]:
outputs_csv_path = PROJECT_ROOT / "outputs" / "pvs_metrics_perc90.csv"

if not outputs_csv_path.is_file():
    raise FileNotFoundError(
        f"Metrics file not found: {outputs_csv_path}"
    )

df_90 = pd.read_csv(outputs_csv_path)


# ADD demographic information of your subjecs to the CSV file 
demo_csv_path = PROJECT_ROOT / "data" / "demographic_data.csv"

if not demo_csv_path.is_file():
    raise FileNotFoundError(
        f"Demographic file not found: {demo_csv_path}. "
        "Please place demographic_data.csv in the data/ directory."
    )

df_demo = pd.read_csv(demo_csv_path)

In [ ]:
# merge the dataset
df_90.rename(columns={"subject_id": "Subject"}, inplace=True)
df_90["Subject"] = df_90["Subject"].astype(str)
df_demo["Subject"] = df_demo["Subject"].astype(str)
merged_df = pd.merge(df_90, df_demo, on="Subject")

## 1) Pairwise Comparisons

## 1-1-1) Evaluation: pvs_to_wm_ratio between EPC and T2w

In [ ]:
paired_df = df_90[ ['pvs_to_wm_ratio_EPC', 'pvs_to_wm_ratio_T2']].dropna()

diff_ratio = (paired_df['pvs_to_wm_ratio_EPC'] - paired_df['pvs_to_wm_ratio_T2'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_to_wm_ratio_EPC'],
        paired_df['pvs_to_wm_ratio_T2']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_to_wm_ratio_EPC'],
        paired_df['pvs_to_wm_ratio_T2']
    )

## 1-1-2) Evaluation: pvs_to_wm_ratio between EPC and T1w

In [ ]:
paired_df = df_90[ ['pvs_to_wm_ratio_EPC', 'pvs_to_wm_ratio_T1']].dropna()

diff_ratio = (paired_df['pvs_to_wm_ratio_EPC'] - paired_df['pvs_to_wm_ratio_T1'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_to_wm_ratio_EPC'],
        paired_df['pvs_to_wm_ratio_T1']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_to_wm_ratio_EPC'],
        paired_df['pvs_to_wm_ratio_T1']
    )

## 1-2-1) Evaluation: pvs_to_wm_contrast_z between EPC and T2w

In [ ]:
paired_df = df_90[ ['pvs_wm_contrast_z_EPC', 'pvs_wm_contrast_z_T2']].dropna()

diff_ratio = (paired_df['pvs_wm_contrast_z_EPC'] - paired_df['pvs_wm_contrast_z_T2'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_wm_contrast_z_EPC'],
        paired_df['pvs_wm_contrast_z_T2']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_wm_contrast_z_EPC'],
        paired_df['pvs_wm_contrast_z_T2']
    )

## 1-2-2) Evaluation: pvs_to_wm_contrast_z between EPC and T1w

In [ ]:
paired_df = df_90[ ['pvs_wm_contrast_z_EPC', 'pvs_wm_contrast_z_T1']].dropna()

diff_ratio = (paired_df['pvs_wm_contrast_z_EPC'] - paired_df['pvs_wm_contrast_z_T1'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_wm_contrast_z_EPC'],
        paired_df['pvs_wm_contrast_z_T1']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_wm_contrast_z_EPC'],
        paired_df['pvs_wm_contrast_z_T1']
    )

## 1-3-1) Evaluation: pvs_count_EPC between EPC and T2w

In [ ]:
paired_df = df_90[ ['pvs_count_EPC', 'pvs_count_T2']].dropna()

diff_ratio = (paired_df['pvs_count_EPC'] - paired_df['pvs_count_T2'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_count_EPC'],
        paired_df['pvs_count_T2']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_count_EPC'],
        paired_df['pvs_count_T2']
    )

## 1-3-2) Evaluation: pvs_count_EPC between EPC and T1w

In [ ]:
paired_df = df_90[ ['pvs_count_EPC', 'pvs_count_T1']].dropna()

diff_ratio = (paired_df['pvs_count_EPC'] - paired_df['pvs_count_T1'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_count_EPC'],
        paired_df['pvs_count_T1']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_count_EPC'],
        paired_df['pvs_count_T1']
    )

## 1-4-1) Evaluation: mean_diameter_vox(_mm) between EPC and T2w

In [ ]:
paired_df = df_90[ ['mean_diameter_mm_EPC', 'mean_diameter_mm_T2']].dropna()

diff_ratio = (paired_df['mean_diameter_mm_EPC'] - paired_df['mean_diameter_mm_T2'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['mean_diameter_mm_EPC'],
        paired_df['mean_diameter_mm_T2']
    )
else:
    stat, pval = wilcoxon(
        paired_df['mean_diameter_mm_EPC'],
        paired_df['mean_diameter_mm_T2']
    )

## 1-4-2) Evaluation: mean_diameter_vox(_mm) between EPC and T1w

In [ ]:
paired_df = df_90[ ['mean_diameter_mm_EPC', 'mean_diameter_mm_T1']].dropna()

diff_ratio = (paired_df['mean_diameter_mm_EPC'] - paired_df['mean_diameter_mm_T1'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['mean_diameter_mm_EPC'],
        paired_df['mean_diameter_mm_T1']
    )
else:
    stat, pval = wilcoxon(
        paired_df['mean_diameter_mm_EPC'],
        paired_df['mean_diameter_mm_T1']
    )

## 1-5-1) Evaluation: PVSVF_wm between EPC and T2w

In [ ]:
paired_df = df_90[ ['pvsvf_wm_EPC', 'pvsvf_wm_T2']].dropna()

diff_ratio = (paired_df['pvsvf_wm_EPC'] - paired_df['pvsvf_wm_T2'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvsvf_wm_EPC'],
        paired_df['pvsvf_wm_T2']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvsvf_wm_EPC'],
        paired_df['pvsvf_wm_T2']
    )

## 1-5-2) Evaluation: PVSVF_wm between EPC and T1w

In [ ]:
paired_df = df_90[ ['pvsvf_wm_EPC', 'pvsvf_wm_T1']].dropna()

diff_ratio = (paired_df['pvsvf_wm_EPC'] - paired_df['pvsvf_wm_T1'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvsvf_wm_EPC'],
        paired_df['pvsvf_wm_T1']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvsvf_wm_EPC'],
        paired_df['pvsvf_wm_T1']
    )

## 1-6-1) Evaluation: pvs_volume between EPC and T2w

In [ ]:
paired_df = df_90[ ['pvs_volume_wm_mm_EPC', 'pvs_volume_wm_mm_T2']].dropna()

diff_ratio = (paired_df['pvs_volume_wm_mm_EPC'] - paired_df['pvs_volume_wm_mm_T2'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_volume_wm_mm_EPC'],
        paired_df['pvs_volume_wm_mm_T2']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_volume_wm_mm_EPC'],
        paired_df['pvs_volume_wm_mm_T2']
    )

## 1-6-2) Evaluation: pvs_volume between EPC and T1w

In [ ]:
paired_df = df_90[ ['pvs_volume_wm_mm_EPC', 'pvs_volume_wm_mm_T1']].dropna()

diff_ratio = (paired_df['pvs_volume_wm_mm_EPC'] - paired_df['pvs_volume_wm_mm_T1'])

stat, pval = shapiro(diff_ratio)

if pval > 0.05:
    stat, pval = ttest_rel(
        paired_df['pvs_volume_wm_mm_EPC'],
        paired_df['pvs_volume_wm_mm_T1']
    )
else:
    stat, pval = wilcoxon(
        paired_df['pvs_volume_wm_mm_EPC'],
        paired_df['pvs_volume_wm_mm_T1']
    )

## 2) Calculate Correlation

## 2-1) Correlation between featurs of EPC & T2

## PVS_count

In [ ]:
x = df_90['pvs_count_EPC'].dropna()
y = df_90['pvs_count_T2'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")


In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## PVS_mean_dim_mm

In [ ]:
x = df_90['mean_diameter_mm_EPC'].dropna()
y = df_90['mean_diameter_mm_T2'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## PVSVF_wm

In [ ]:
x = df_90['pvsvf_wm_EPC'].dropna()
y = df_90['pvsvf_wm_T2'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## pvs_wm_contrast_z

In [ ]:
x = df_90['pvs_wm_contrast_z_EPC'].dropna()
y = df_90['pvs_wm_contrast_z_T2'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## PVS Volume

In [ ]:
x = df_90['pvs_volume_wm_mm_EPC'].dropna()
y = df_90['pvs_volume_wm_mm_T2'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## pvs_to_wm_ratio

In [ ]:
x = df_90['pvs_to_wm_ratio_EPC'].dropna()
y = df_90['pvs_to_wm_ratio_T2'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## 2-2) Correlation between featurs of EPC & T1

## PVS_count

In [ ]:
x = df_90['pvs_count_EPC'].dropna()
y = df_90['pvs_count_T1'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T1:  p = {p_y:.3f}")


In [ ]:

if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.23}, p = {pval:.3e}")



## PVS_mean_dim_vox

In [ ]:

x = df_90['mean_diameter_vox_EPC'].dropna()
y = df_90['mean_diameter_vox_T1'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T1:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## PVSVF_wm

In [ ]:
x = df_90['pvsvf_wm_EPC'].dropna()
y = df_90['pvsvf_wm_T1'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T1:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## pvs_wm_contrast_z

In [ ]:
x = df_90['pvs_wm_contrast_z_EPC'].dropna()
y = df_90['pvs_wm_contrast_z_T1'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T1:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## PVS volume

In [ ]:
x = df_90['pvs_volume_wm_mm_EPC'].dropna()
y = df_90['pvs_volume_wm_mm_T1'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

## pvs_to_wm_ratio

In [ ]:
x = df_90['pvs_to_wm_ratio_EPC'].dropna()
y = df_90['pvs_to_wm_ratio_T1'].dropna()

# Make sure they align
paired_df = pd.DataFrame({'x': x, 'y': y}).dropna()

# Shapiro-Wilk tests
_, p_x = shapiro(paired_df['x'])
_, p_y = shapiro(paired_df['y'])

print(f"Shapiro EPC: p = {p_x:.3f}")
print(f"Shapiro T2:  p = {p_y:.3f}")

In [ ]:
if p_x > 0.05 and p_y > 0.05:
    r, pval = pearsonr(paired_df['x'], paired_df['y'])
    print(f"Pearson correlation: r = {r:.3f}, p = {pval:.3e}")
else:
    r, pval = spearmanr(paired_df['x'], paired_df['y'])
    print(f"Spearman correlation: r = {r:.3f}, p = {pval:.3e}")

In [ ]:
# 	Show the distribution of observed measurements across modalities

def plot_modality_distribution(df, variable, title=None, palette="Set2", save_path=None):
    """
    Plot and save the distribution of a variable across modalities.

    Parameters:
    - df: DataFrame with 'modality' and the variable.
    - variable: str, variable name.
    - title: str, plot title (optional).
    - palette: str or list, seaborn palette.
    - save_path: str, if provided, saves the plot to this filepath.
    """


    sns.set(style="whitegrid", font_scale=1.2)
    plt.figure(figsize=(8, 6))

    sns.boxplot(x="modality", y=variable, data=df, palette=palette)
    sns.stripplot(x="modality", y=variable, data=df, color="black", alpha=0.3, jitter=True)

    plt.title(f"{variable} across Modalities")
    plt.xlabel("Modality")
    plt.ylabel(variable)
    plt.tight_layout()

    plt.savefig(save_path, dpi=300, bbox_inches="tight", format="png")

    plt.show()


In [ ]:
# Show conditional fitted values from the mixed-effects model

def plot_lme_predictions(df_clean, model, variable, title=None, save_path=None):
    """
    Plot conditional fitted values from the LME model across modalities.
    These fitted values include the estimated subject-specific random effects.
    """

    sns.set(style="whitegrid", font_scale=1.2)
    df_clean = df_clean.copy()
    df_clean["fitted"] = model.fittedvalues

    plt.figure(figsize=(8, 6))
    sns.boxplot(x="modality", y="fitted", data=df_clean, palette="Set2")
    sns.stripplot(x="modality", y="fitted", data=df_clean, color="black", alpha=0.3, jitter=True)

    plt.title(title or f"Conditional Fitted Values of {variable}", fontsize=12)
    plt.ylabel(f"Conditional Fitted {variable}", fontsize=10)
    plt.xlabel("Modality")
    plt.tight_layout()


    plt.savefig(save_path, dpi=300, format="png", bbox_inches="tight")

    plt.show()


In [ ]:
# Evaluate model residual behavior

def plot_model_diagnostics(model):
    """
    Generate diagnostic plots for a fitted linear mixed-effects model:
    1. Q-Q plot of residuals
    2. Residuals versus fitted values
    """

    residuals = model.resid
    fitted = model.fittedvalues

    # Q-Q plot of residuals
    plt.figure(figsize=(6, 6))
    stats.probplot(residuals, dist="norm", plot=plt)
    plt.title("QQ Plot of Residuals")
    plt.tight_layout()
    plt.show()

    # Residuals versus fitted values
    plt.figure(figsize=(8, 6))
    plt.scatter(fitted, residuals, alpha=0.5)
    plt.axhline(0, color="red", linestyle="--")
    plt.xlabel("Fitted Values")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Fitted Values")
    plt.tight_layout()
    plt.show()

In [ ]:
def run_lme(df_merged, variable):

    long_df = pd.wide_to_long(
        df_merged,
        stubnames=[variable],
        i="Subject",
        j="modality",
        sep="_",
        suffix="(T1|T2|EPC)"
    ).reset_index()

    # Remove incomplete rows
    df_clean = long_df.dropna(
        subset=[variable, "Age", "Gender", "FS_IntraCranial_Vol"]
    ).copy()

    # Set modality as categorical with T2 as the reference
    df_clean["modality"] = pd.Categorical(
        df_clean["modality"],
        categories=["T2", "T1", "EPC"]
    )

    # Fit the linear mixed-effects model
    model = smf.mixedlm(
        f"{variable} ~ modality + C(Age) + C(Gender) + FS_IntraCranial_Vol",
        data=df_clean,
        groups=df_clean["Subject"]
    ).fit()

    print(model.summary())

    # Check model convergence and random effects
    print("Model converged:", model.converged)

    print("\nRandom-effects covariance:")
    print(model.cov_re)

    print("\nResidual variance:")
    print(model.scale)

    # Plot the distribution across modalities
    plot_modality_distribution(
        df_clean,
        variable=variable,
        title=f"{variable.replace('_', ' ').title()} Across Modalities",
        save_path=f"./results/{variable}_distribution.png"
    )

    # Plot conditional fitted values
    metric_title = variable.replace("_", " ").title()

    plot_lme_predictions(
        df_clean,
        model,
        variable=variable,
        title=f"Conditional Fitted Values of {metric_title}",
        save_path=f"./results/{variable}_lme_pred.png"
    )

    # Model diagnostic plots
    print(f"\nModel diagnostics for: {variable}")
    plot_model_diagnostics(model)

    return model, df_clean

In [ ]:
run_lme(merged_df, "pvs_volume_wm_mm")
run_lme(merged_df, "pvs_count")
run_lme(merged_df, "mean_diameter_mm")
run_lme(merged_df, "pvs_wm_contrast_z")
run_lme(merged_df, "pvs_to_wm_ratio")
run_lme(merged_df, "pvsvf_wm")

In [ ]:
# Check values for the suspicious metric
print(df_90['pvsvf_wm_T1'].describe())
print(df_90['pvsvf_wm_T1'].unique())

In [ ]:
metrics = ['pvs_volume_wm_mm', 'mean_diameter_mm', 'pvs_count', 'pvsvf_wm', 'pvs_to_wm_ratio', 'pvs_wm_contrast_z']
modalities = ['T1', 'T2', 'EPC']

In [ ]:
summary_data = []

for metric in metrics:
    row = {'Metric': metric.replace('_', ' ').title()}
    for mod in modalities:
        col_name = f"{metric}_{mod}"
        if col_name in df_90.columns:
            mean = df_90[col_name].mean()
            std = df_90[col_name].std()
            row[f"{mod}w Mean ± SD"] = f"{mean:.3f} ± {std:.3f}"
        else:
            row[f"{mod}w Mean ± SD"] = "—"
    summary_data.append(row)

# Convert to DataFrame
summary_df = pd.DataFrame(summary_data)
print(summary_df)
